# House Price Prediction Using Data Science and Machine Learning

**Internship project | Linear Regression | Google Colab**

This notebook covers data loading, cleaning, EDA, train/test split, Linear Regression, prediction and evaluation.

**Dataset:** King County House Sales (`kc_house_data.csv`)  
Download it from [Kaggle – House Sales in King County](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction), then upload the CSV when prompted below.


In [ ]:
# 1. Import libraries
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")


## 2. Upload and load the dataset
Run this cell in Google Colab and select `kc_house_data.csv`.

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload the dataset CSV file.")

df = pd.read_csv(csv_files[0])
print("Loaded:", csv_files[0])
display(df.head())


## 3. Understand the dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumns:")
print(df.columns.tolist())
print("\\nDataset information:")
df.info()
print("\\nStatistical summary:")
display(df.describe())


## 4. Clean the data

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates().copy()

# Remove identifier/date columns when present
df = df.drop(columns=["id", "date"], errors="ignore")

# Convert any infinite values to missing, then remove rows with missing values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna().copy()

if "price" not in df.columns:
    raise ValueError("The dataset must contain a 'price' column.")

print("Cleaned dataset shape:", df.shape)
print("Missing values remaining:", int(df.isnull().sum().sum()))


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# House price distribution
plt.figure(figsize=(10, 5))
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("House price ($)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
# Living area vs house price (if available)
if "sqft_living" in df.columns:
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=df, x="sqft_living", y="price", alpha=0.5)
    plt.title("Living Area vs House Price")
    plt.xlabel("Living area (sq. ft.)")
    plt.ylabel("House price ($)")
    plt.tight_layout()
    plt.show()
else:
    print("sqft_living column not found; skipping this plot.")


In [ ]:
# Bedrooms vs house price (if available)
if "bedrooms" in df.columns:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, x="bedrooms", y="price")
    plt.title("Bedrooms vs House Price")
    plt.xlabel("Number of bedrooms")
    plt.ylabel("House price ($)")
    plt.tight_layout()
    plt.show()
else:
    print("bedrooms column not found; skipping this plot.")


In [ ]:
# Correlation heatmap for numeric columns
numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
# Average price by number of bedrooms (if available)
if "bedrooms" in df.columns:
    average_price = df.groupby("bedrooms")["price"].mean()
    plt.figure(figsize=(9, 5))
    average_price.plot(kind="bar")
    plt.title("Average House Price by Bedrooms")
    plt.xlabel("Number of bedrooms")
    plt.ylabel("Average price ($)")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 6. Select input features (X) and target (y)

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

# Linear Regression requires numeric input features.
# This dataset is expected to contain numeric features after dropping id/date.
X = X.select_dtypes(include=np.number)

if X.shape[1] == 0:
    raise ValueError("No numeric input features found.")

print("Input feature columns:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)


## 7. Split into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


## 8. Train the Linear Regression model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")


## 9. Predict house prices

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": y_pred
})
display(results.head(10))


## 10. Evaluate the model

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MODEL EVALUATION RESULTS")
print(f"MAE  : ${mae:,.2f}")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : ${rmse:,.2f}")
print(f"R²   : {r2:.4f} ({r2 * 100:.2f}%)")


## 11. Actual vs predicted price plot

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual house price ($)")
plt.ylabel("Predicted house price ($)")
plt.title("Actual vs Predicted House Prices")
plt.tight_layout()
plt.show()


## 12. Predict the price of a new house
The example below starts with median values for all model features and changes selected features when those columns exist.

In [ ]:
new_house = X.median(numeric_only=True).to_frame().T

example_values = {
    "bedrooms": 3,
    "bathrooms": 2,
    "sqft_living": 1500,
    "sqft_lot": 5000,
    "floors": 1
}

for feature, value in example_values.items():
    if feature in new_house.columns:
        new_house.loc[:, feature] = value

new_house = new_house[X.columns]
new_price = model.predict(new_house)[0]

print("New house input:")
display(new_house)
print(f"Estimated house price: ${new_price:,.2f}")


## 13. Inspect Linear Regression coefficients

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

display(coefficients)


In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(data=coefficients, x="Coefficient", y="Feature")
plt.title("Linear Regression Coefficients")
plt.tight_layout()
plt.show()


## 14. Save results for your report

In [ ]:
os.makedirs("outputs", exist_ok=True)

results.to_csv("outputs/house_price_predictions.csv", index=False)

evaluation = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Value": [mae, mse, rmse, r2]
})
evaluation.to_csv("outputs/model_evaluation.csv", index=False)

coefficients.to_csv("outputs/model_coefficients.csv", index=False)

print("Saved prediction, evaluation and coefficient CSV files in outputs/.")


## 15. Conclusion

This project used Python libraries to load and clean house-sales data, explore it through visualizations, train a Linear Regression model, predict house prices, and evaluate model performance using MAE, MSE, RMSE and R².

**Important:** Report the actual metric values produced when you run the notebook. Predictions are estimates and not guaranteed market prices.
